# InvoiceGuard Round 2 -- Training Reproducibility Notebook

**Hackathon:** OpenEnv Hackathon 2026 (Round 2)  
**Author:** piyush-mk  
**Model:** Qwen/Qwen3-4B-Instruct-2507 (bf16 LoRA)

This notebook reproduces the InvoiceGuard training pipeline end-to-end. It trains a LoRA adapter using **submit-focused SFT** -- the technique that achieved **0.729 grader score** (5.3× improvement over the untrained local baseline).

**No HF Space or external job dependency.** All training code is pulled from `piyush-mk/invoiceguard-code` on the Hub. Trained adapters and artifacts are pushed back to the Hub.

### How to Run
1. Open in **Colab** (GPU runtime, T4 or better) or **Kaggle** (GPU P100/T4)
2. Set `HF_TOKEN` in Secrets (needs write access)
3. Run all cells top-to-bottom
4. Check the output repo on Hub for adapter + metrics

In [ ]:
%pip -q install -U uv huggingface_hub

In [ ]:
import os

# Auto-detect HF_TOKEN from Colab or Kaggle secrets
try:
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN") or os.environ.get("HF_TOKEN", "")
except Exception:
    pass
try:
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN") or os.environ.get("HF_TOKEN", "")
except Exception:
    pass

os.environ.setdefault("HF_USERNAME", "piyush-mk")
os.environ.setdefault("INVOICEGUARD_CODE_REPO", "piyush-mk/invoiceguard-code")
os.environ.setdefault("BASE_MODEL", "Qwen/Qwen3-4B-Instruct-2507")
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

assert os.environ.get("HF_TOKEN"), "Set HF_TOKEN before running."
print(f"Model: {os.environ['BASE_MODEL']}")
print(f"Code repo: {os.environ['INVOICEGUARD_CODE_REPO']}")

In [ ]:
import torch
print(f"torch {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))
    free, total = torch.cuda.mem_get_info()
    print(f"VRAM: {free/1e9:.1f} / {total/1e9:.1f} GB")

## 1. Submit-Focused SFT (Recommended)

This is the training approach that achieved **0.729 grader score**. It trains only on `submit_final_resolution` examples from traces with 7+ investigation steps.

Key flags:
- `--no-4bit`: Use bf16 (critical -- 4-bit quantization degrades all results)
- `--submit-only`: Train only on submit examples
- `--min-investigation-steps 7`: Only use traces with deep investigation context

In [ ]:
!HF_TOKEN="$HF_TOKEN" \
HF_USERNAME="$HF_USERNAME" \
INVOICEGUARD_CODE_REPO="$INVOICEGUARD_CODE_REPO" \
BASE_MODEL="$BASE_MODEL" \
HUB_MODEL_ID="invoiceguard-qwen3-4b-sft-notebook" \
TRACKIO_PROJECT="invoiceguard-round2" \
TRACKIO_RUN_NAME="sft-notebook" \
PYTORCH_CUDA_ALLOC_CONF="$PYTORCH_CUDA_ALLOC_CONF" \
TOKENIZERS_PARALLELISM="$TOKENIZERS_PARALLELISM" \
uv run https://huggingface.co/$INVOICEGUARD_CODE_REPO/resolve/main/training/train_sft.py \
  --no-4bit \
  --submit-only \
  --min-investigation-steps 7 \
  --num-epochs 12 \
  --eval-holdout-canonical 2 \
  --eval-holdout-hard 2 \
  --lr 5e-5 \
  --max-new-tokens 384 \
  --max-prompt-tokens 2048

## 2. Verify Results

Check that the adapter and training artifacts landed on the Hub.

In [ ]:
from huggingface_hub import HfApi, hf_hub_download
import json

REPO = f"{os.environ['HF_USERNAME']}/invoiceguard-qwen3-4b-sft-notebook"
api = HfApi(token=os.environ["HF_TOKEN"])

files = api.list_repo_files(repo_id=REPO, repo_type="model")
print(f"Repo: {REPO}")
print(f"Files: {len(files)}")
for f in sorted(files):
    print(f"  {f}")

# Show metrics
try:
    metrics_path = hf_hub_download(REPO, "sft_artifacts/sft_metrics.jsonl", token=os.environ["HF_TOKEN"])
    print("\n--- Eval Results ---")
    with open(metrics_path) as f:
        for line in f:
            row = json.loads(line)
            if "eval/avg_grader_score" in row:
                print(f"Epoch {row['step']:2d}: score={row['eval/avg_grader_score']:.4f}  "
                      f"success={row['eval/success_rate']:.0%}  "
                      f"steps={row['eval/avg_steps']:.1f}")
except Exception as e:
    print(f"Could not load metrics: {e}")

## 3. GRPO (Optional)

Run GRPO starting from the SFT adapter above. This uses reinforcement learning to further refine the model's decisions.

In [ ]:
# Uncomment to run GRPO from the SFT checkpoint
# !HF_TOKEN="$HF_TOKEN" \
# HF_USERNAME="$HF_USERNAME" \
# INVOICEGUARD_CODE_REPO="$INVOICEGUARD_CODE_REPO" \
# BASE_MODEL="$BASE_MODEL" \
# HUB_MODEL_ID="invoiceguard-qwen3-4b-grpo-notebook" \
# TRACKIO_PROJECT="invoiceguard-round2" \
# TRACKIO_RUN_NAME="grpo-notebook" \
# PYTORCH_CUDA_ALLOC_CONF="$PYTORCH_CUDA_ALLOC_CONF" \
# TOKENIZERS_PARALLELISM="$TOKENIZERS_PARALLELISM" \
# uv run https://huggingface.co/$INVOICEGUARD_CODE_REPO/resolve/main/training/train_grpo.py \
#   --no-4bit \
#   --no-format-warmup \
#   --resume-adapter $HF_USERNAME/invoiceguard-qwen3-4b-sft-notebook \
#   --num-iterations 3 \
#   --group-size 8 \
#   --sample-temperature 1.3 \
#   --eval-holdout-canonical 2 \
#   --eval-holdout-hard 2 \
#   --max-new-tokens 384 \
#   --max-prompt-tokens 2048

## Expected Output

A successful run uploads to the Hub:

| Artifact | File |
|----------|------|
| LoRA adapter | `adapter_model.safetensors` + `adapter_config.json` |
| Training metrics | `sft_artifacts/sft_metrics.jsonl` |
| Summary | `sft_artifacts/sft_summary.json` |

The metrics JSONL contains per-epoch eval scores. Look for `eval/avg_grader_score` rising from ~0.14 (untrained) to 0.6-0.7+ (trained).